# pyPetrograph — multimodal grain demo (v0.0.6)

One PPL + XPL pair from `Test images/` (default AV03). Align → physics channels → SegmentEveryGrain **U-Net** outlines → object table → cluster / name → LightGBM + stats.

**U-Net is enough** for grain counts and the object table (~26 MB weights). SAM 2.1 refine is only for finer grain *shapes* and downloads ~860 MB — leave `USE_SAM2 = False` unless you need that.

`INTERACTIVE_QC = False` writes grains without a GrainPlot window. `INTERACTIVE_NAMES = False` skips the montage labeler (step 6 then stops at cluster ids). If SegmentEveryGrain / TensorFlow is missing, step 3 uses the watershed outlines instead.

Outputs go to `outputs/` (`align/`, `channels/`, `labels/`, `objects/`, `models/`).


## Setup

In [ ]:
from pathlib import Path
import sys
ROOT = Path(".")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

from pyPetrograph import check_and_install_packages
check_and_install_packages(include_seg=True)


In [ ]:
%matplotlib inline

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 150

from pyPetrograph import (
    set_output_dir,
    Scene,
    launch_align,
    save_scene,
    load_scene,
    DEFAULT_CHANNEL_TOGGLES,
    build_channel_stack,
    save_channel_stack,
    load_channel_stack,
    run_channel_preview,
    run_watershed,
    launch_ppl_grain_qc,
    launch_grain_qc,
    load_grain_qc,
    load_grain_mask,
    grain_qc_paths,
    overlay_labels,
    compare_outlines,
    ensure_seg_weights,
    build_object_table_from_stack,
    save_object_table,
    load_object_table,
    cluster_objects,
    launch_montage_labeler,
    load_object_labels,
    apply_object_labels,
    train_object_classifier,
    predict_object_classes,
    save_classes,
    compute_object_stats,
    save_object_stats,
)
from pyPetrograph.align.io import default_json_path


## Settings

This demo uses one PPL + XPL pair (AV03).

- `USE_SAM2` — extra shape refine (off; large download)
- `INTERACTIVE_QC` / `INTERACTIVE_NAMES` — pop-up windows (off here)
- `LOAD_SAVED_*` — reuse files already in `outputs/`


In [ ]:
set_output_dir("outputs")

IMAGE_DIR = Path("Test images")
# Also in Test images: AV11_ppl_10x_pic04t4 + AV11_xpl_…, AV12_ppl_10x_pic03t3 + AV12_xpl_…
STEM = "AV03_ppl_10x_pic03t3"
PPL = IMAGE_DIR / f"{STEM}.tif"
XPL = IMAGE_DIR / "AV03_xpl_10x_pic03t3.tif"

WORKING_SLOT = "ppl"
OVERLAY_SLOT = "xpl"

OUTLINE_ENGINE = "seg"       # "seg" | "watershed"
USE_SAM2 = False             # True = SAM shape refine (~860 MB). Leave False.
INTERACTIVE_QC = False       # True opens GrainPlot
INTERACTIVE_NAMES = False    # True opens the montage labeler

EXTRA_TEXTURE = False
EXTRA_EMBED = False
CHANNEL_TOGGLES = dict(DEFAULT_CHANNEL_TOGGLES)
CHANNEL_TOGGLES["extra_texture"] = EXTRA_TEXTURE
CHANNEL_TOGGLES["extra_embed"] = EXTRA_EMBED

CLASS_NAMES = {1: "pores", 2: "grains", 3: "cements"}
K_CLUSTERS = 8
PX_PER_UM = None

LOAD_SAVED_ALIGN = True
LOAD_SAVED_CHANNELS = True
LOAD_SAVED_GRAINS = True
LOAD_SAVED_TABLE = True
LOAD_SAVED_LABELS = True
LOAD_SAVED_CLASSES = True

INIT_UNET = False          # True = rebuild N-channel net from SEG RGB weights
TRAIN_UNET = False         # True = train again even if grain_unet_*.trained.json exists
UNET_EPOCHS = 15

scene = Scene(working_slot=WORKING_SLOT)
scene.add_layer("ppl", PPL, is_working=True)
scene.add_layer("xpl", XPL)
print(PPL.name, "+", XPL.name)
print("outline:", OUTLINE_ENGINE, "SAM2:", USE_SAM2, "qc:", INTERACTIVE_QC, "names:", INTERACTIVE_NAMES)


## 1. Align

Shift the XPL photo so it sits on the same pixels as the PPL photo. `tx` / `ty` are that shift in pixels (right / down).


In [ ]:
from pyPetrograph.align.register import auto_register

align_json = default_json_path(scene)
if LOAD_SAVED_ALIGN and align_json.exists():
    scene = load_scene(align_json)
    print("loaded", align_json)
else:
    base = scene.working().ensure_rgb()
    for slot, layer in scene.layers.items():
        if slot == scene.working_slot:
            continue
        layer.transform = auto_register(
            base,
            layer.ensure_rgb(),
            scale_hint=layer.transform.scale,
            scale_range=scene.scale_range,
            tx_hint=layer.transform.tx,
            ty_hint=layer.transform.ty,
        )
        print(slot, layer.transform)
    save_scene(scene)
    print("saved", scene.json_path)


In [ ]:
# Optional tweak window (skip for a headless demo)
# launch_align(scene, overlay_slot=OVERLAY_SLOT, wait=False)


In [ ]:
scene = load_scene(default_json_path(scene) if scene.json_path is None else scene.json_path)
print("loaded", scene.json_path)
for slot, layer in scene.layers.items():
    print(slot, layer.transform)


## 2. Measurement channels

One map per measurement, already on the PPL grid. Here: PPL red / green / blue and XPL red / green / blue.


In [ ]:
cs = load_channel_stack(PPL) if LOAD_SAVED_CHANNELS else None
if cs is None:
    cs = build_channel_stack(scene, toggles=CHANNEL_TOGGLES)
    save_channel_stack(cs, PPL)
    print("built and saved")
else:
    print("loaded saved stack")
print(cs.summary())


In [ ]:
run_channel_preview(cs, image_path=PPL)

## 3. Grain outlines

The demo first traces grains with **watershed** (no training). [SegmentEveryGrain](https://github.com/zsylvester/segmenteverygrain) is optional: its U-Net can refine those outlines. GrainPlot QC is a further optional window (`INTERACTIVE_QC`).


In [ ]:
ws_lab, ws_info = run_watershed(cs)
print("watershed", ws_info)
ppl_rgb = scene.working().ensure_rgb()


In [ ]:
from PIL import Image

def _write_ws_mask(image_path, lab):
    _geo, mask_p = grain_qc_paths(image_path)
    mask_p.parent.mkdir(parents=True, exist_ok=True)
    lab = np.asarray(lab)
    if int(lab.max()) <= 255:
        Image.fromarray(lab.astype(np.uint8), mode="L").save(mask_p)
    else:
        Image.fromarray(lab.astype(np.uint16), mode="I;16").save(mask_p)
    n = int(np.sum(np.unique(lab) > 0))
    print(f"wrote {mask_p.name} n={n}")
    return {"ok": True, "n": n, "mask": mask_p, "engine": "watershed"}

if OUTLINE_ENGINE == "seg":
    print(ensure_seg_weights(sam2=USE_SAM2))

if LOAD_SAVED_GRAINS and load_grain_mask(PPL) is not None:
    qc = load_grain_qc(PPL)
    print("loaded grains", qc)
elif OUTLINE_ENGINE == "seg":
    qc = launch_ppl_grain_qc(
        PPL, reuse=False, use_sam=USE_SAM2, interactive=INTERACTIVE_QC
    )
    print(qc)
    if not qc.get("ok"):
        print("SEG / GrainPlot failed — using watershed outlines")
        qc = _write_ws_mask(PPL, ws_lab)
else:
    qc = _write_ws_mask(PPL, ws_lab)


### Without SEG (watershed only)

No trained grain finder. Bright/dark basins are split by watershed. This is the default first guess.


In [ ]:
fig = compare_outlines(ppl_rgb, watershed_lab=ws_lab, greyscale=True)
plt.show()


### SEG-assisted outlines

Left: watershed (no SEG). Right: saved [SegmentEveryGrain](https://github.com/zsylvester/segmenteverygrain) U-Net mask (`outputs/labels/{stem}_grains_mask.png`). That right panel is SEG-assisted, not watershed. GrainPlot was not opened in this demo (`INTERACTIVE_QC = False`).


In [ ]:
qc_lab = load_grain_mask(PPL)
fig = compare_outlines(
    ppl_rgb,
    watershed_lab=ws_lab,
    qc_lab=qc_lab,
    greyscale=True,
    titles=("watershed (no SEG)", "SEG U-Net", "SEG-assisted U-Net"),
)
plt.show()


## Optional: N-channel U-Net

Trains once from the SEG grain mask (`_grains_mask.png`), using the PPL + XPL stack. If `outputs/models/grain_unet_{channel_set}.keras` plus `.trained.json` already exist, this cell **loads** the model and predicts — it does not train again.

Set `INIT_UNET` / `TRAIN_UNET` True in Settings only to force a redo.


In [ ]:
from pyPetrograph import (
    init_grain_unet,
    train_grain_unet,
    predict_grain_unet,
    grain_unet_path,
    grain_unet_is_trained,
    load_grain_mask,
)

weights = grain_unet_path(cs.channel_set_id)
already = grain_unet_is_trained(cs.channel_set_id)

if INIT_UNET or not weights.exists():
    print(init_grain_unet(cs))
if TRAIN_UNET or not already:
    if load_grain_mask(PPL) is None:
        print("no SEG grain mask — skip N-channel train")
    else:
        print(train_grain_unet(cs, PPL, epochs=UNET_EPOCHS))
else:
    print(f"loaded trained U-Net {weights.name} (skip train)")

if grain_unet_path(cs.channel_set_id).exists():
    unet_probs, unet_lab, unet_info = predict_grain_unet(cs)
    print("U-Net", unet_info)
    if unet_lab is not None:
        fig = compare_outlines(
            ppl_rgb,
            watershed_lab=ws_lab,
            unet_lab=unet_lab,
            qc_lab=load_grain_mask(PPL),
            greyscale=True,
            titles=("watershed (no SEG)", "N-channel U-Net", "SEG mask (train target)"),
        )
        plt.show()
else:
    print("no N-channel grain U-Net yet")


## 4. Object table

One row per region.

- **grain** — a U-Net polygon
- **leftover** — the space between grains (pores, cement, or matrix)

`table.head()` shows the first rows only.


In [ ]:
loaded = load_object_table(PPL) if LOAD_SAVED_TABLE else None
if loaded is not None:
    table, objects_mask = loaded
    print("loaded table", len(table))
else:
    grains = load_grain_mask(PPL)
    if grains is None:
        raise RuntimeError("no grain mask — run step 3")
    table, objects_mask = build_object_table_from_stack(
        grains, cs, px_per_um=PX_PER_UM
    )
    save_object_table(table, objects_mask, PPL)
    print("built table", len(table))
print(table["kind"].value_counts().to_string() if "kind" in table.columns else table.head())
table.head()


## 5. Group look-alikes

Puts similar objects into 8 groups (`cluster_id`). This demo does not open the naming window, so groups stay as numbers.


In [ ]:
table = cluster_objects(table, n_clusters=K_CLUSTERS)
blob = load_object_labels(PPL) if LOAD_SAVED_LABELS else None
if blob is None and INTERACTIVE_NAMES:
    print(launch_montage_labeler(PPL, table, objects_mask, CLASS_NAMES))
    blob = load_object_labels(PPL)
if blob is None:
    print("no names yet — set INTERACTIVE_NAMES = True to open the montage, or skip classify")
else:
    table = apply_object_labels(table, blob)
    print("named", int((table["class_id"] > 0).sum()), "of", len(table))


## 6. Classify and stats

Needs named examples from step 5. Skipped here because names were not assigned.


In [ ]:
from pyPetrograph import print_train_metrics, object_model_path, load_classes
from pyPetrograph.labeling_ml.model_io import load_model_bundle

model_p = object_model_path(cs.channel_set_id)
if LOAD_SAVED_CLASSES and model_p.exists() and load_classes(PPL) is not None:
    table = predict_object_classes(table, load_model_bundle(model_p))
    print("loaded model", model_p.name)
elif blob is not None:
    train_res = train_object_classifier(
        table, CLASS_NAMES, channel_set_id=cs.channel_set_id
    )
    print(train_res.get("error") or f"trained n={train_res.get('n')}")
    if train_res.get("ok"):
        print_train_metrics(train_res["bundle"])
        table = predict_object_classes(table, train_res["bundle"])
else:
    print("skip classify — name some clusters in step 5 first")

if "class_id" in table.columns and int((table["class_id"] > 0).sum()):
    csv_p, png_p = save_classes(table, objects_mask, PPL, CLASS_NAMES)
    print("wrote", csv_p.name, png_p.name)
    plt.figure(figsize=(7, 5), dpi=150)
    plt.imshow(plt.imread(png_p))
    plt.axis("off")
    plt.title("object classes")
    plt.show()
    stats = compute_object_stats(table, CLASS_NAMES)
    save_object_stats(stats, PPL)
    display(stats)
